# WoundSense — UNet ONNX Model Generation

This notebook trains (or exports) the wound segmentation UNet to ONNX format.

**Options:**
1. **Quick demo**: Export a random-weight UNet to ONNX (test pipeline)
2. **Fine-tune**: Load ImageNet pre-trained ResNet-34 encoder + train on wound data
3. **Full training**: Train on MICCAI wound dataset (requires ~4h on T4)

**Research context:**
- Ronneberger et al. U-Net, MICCAI 2015 — 89% Dice (Jupyter only)
- WoundAmbit arXiv 2025 — 92% Dice (lab prototype)
- WoundSense target: **93% Dice**, production ONNX, <2s inference

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q segmentation-models-pytorch onnx onnxruntime torch torchvision

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import onnx
import onnxruntime as ort
from pathlib import Path

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# ── 1. Define UNet architecture ───────────────────────────────────────────────
# Uses segmentation_models_pytorch for production-ready UNet with ResNet-34 encoder

import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',   # Transfer learning
    in_channels=3,
    classes=1,                    # Binary: wound / background
    activation=None,              # Raw logits; sigmoid applied in post-processing
)
model = model.to(DEVICE)
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')
print(f'Model size estimate: ~{sum(p.numel() * 4 for p in model.parameters()) / 1e6:.1f} MB')

In [ ]:
# ── 2. (Optional) Training loop ───────────────────────────────────────────────
# Skip this cell and go to Export if you just want the demo ONNX.
# For real training: mount Google Drive and point DATASET_DIR at wound images.

TRAIN_MODEL = False  # Set True to train

if TRAIN_MODEL:
    from torch.utils.data import Dataset, DataLoader
    import cv2
    import albumentations as A
    from albumentations.pytorch import ToTensorV2

    # ── Dataset ──────────────────────────────────────────────────────────────
    class WoundDataset(Dataset):
        def __init__(self, image_dir, mask_dir, transform=None):
            self.images = sorted(Path(image_dir).glob('*.jpg'))
            self.mask_dir = Path(mask_dir)
            self.transform = transform

        def __len__(self): return len(self.images)

        def __getitem__(self, idx):
            img_path = self.images[idx]
            mask_path = self.mask_dir / img_path.name.replace('.jpg', '_mask.png')
            img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            if self.transform:
                aug = self.transform(image=img, mask=mask)
                img, mask = aug['image'], aug['mask']
            return img, (mask > 128).float().unsqueeze(0)

    aug = A.Compose([
        A.Resize(512, 512),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.3),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2(),
    ])

    # Mount your wound dataset here:
    DATASET_DIR = '/content/drive/MyDrive/wound_dataset'
    dataset = WoundDataset(f'{DATASET_DIR}/images', f'{DATASET_DIR}/masks', aug)
    loader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=2)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = smp.losses.DiceLoss(mode='binary') + smp.losses.SoftBCEWithLogitsLoss()

    for epoch in range(30):
        model.train()
        total_loss = 0
        for imgs, masks in loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            preds = model(imgs)
            loss = criterion(preds, masks)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f'Epoch {epoch+1}/30 — Loss: {total_loss/len(loader):.4f}')

    print('Training complete!')

In [ ]:
# ── 3. Export to ONNX ─────────────────────────────────────────────────────────

model.eval()
dummy_input = torch.randn(1, 3, 512, 512).to(DEVICE)

onnx_path = '/content/wound_unet.onnx'

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

# Validate
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
size_mb = Path(onnx_path).stat().st_size / 1e6
print(f'✅ ONNX exported: {onnx_path} ({size_mb:.1f} MB)')

In [ ]:
# ── 4. Benchmark ONNX inference time ─────────────────────────────────────────
import time

sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
dummy_np = np.random.randn(1, 3, 512, 512).astype(np.float32)

# Warmup
for _ in range(3):
    sess.run(None, {'input': dummy_np})

# Benchmark
times = []
for _ in range(20):
    t = time.monotonic()
    sess.run(None, {'input': dummy_np})
    times.append((time.monotonic() - t) * 1000)

print(f'Inference time (CPU): {np.mean(times):.0f}ms ± {np.std(times):.0f}ms')
print(f'Note: Snapdragon 665 (Moto G32) ~1.8x slower than Colab CPU → expect ~{np.mean(times)*1.8:.0f}ms')

In [ ]:
# ── 5. Export tissue_classifier.pkl (HSV rule-based) ─────────────────────────
import pickle

# The production classifier is the rule-based HSV thresholder in tissue_classifier.py
# This pickle just stores calibrated thresholds as a config dict for the backend

HSV_THRESHOLDS = {
    'version': '1.0',
    'granulation': {'h_range': [(0, 20), (160, 180)], 's_min': 80, 'v_min': 60},
    'slough':      {'h_range': [(20, 40)],             's_min': 30, 'v_min': 130},
    'necrotic':    {'h_range': [(0, 180)],             's_max': 255, 'v_max': 50},
    'epithelial':  {'h_range': [(140, 175)],           's_min': 10, 's_max': 60, 'v_min': 160},
    'calibration': 'PHC ambient lighting, 2024-11',
}

pkl_path = '/content/tissue_classifier.pkl'
with open(pkl_path, 'wb') as f:
    pickle.dump(HSV_THRESHOLDS, f)

print(f'✅ tissue_classifier.pkl saved to {pkl_path}')

# Download both files
from google.colab import files
files.download('/content/wound_unet.onnx')
files.download('/content/tissue_classifier.pkl')